## Import tools

In [52]:
import numpy as np
import pandas as pd
import seaborn as sns


## Get the data

In [53]:
d = pd.read_csv("dataset.csv")
d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   loan_id            614 non-null    object 
 1   gender             601 non-null    object 
 2   married            611 non-null    object 
 3   dependents         599 non-null    object 
 4   education          614 non-null    object 
 5   self_employed      582 non-null    object 
 6   applicantincome    614 non-null    int64  
 7   coapplicantincome  614 non-null    float64
 8   loanamount         592 non-null    float64
 9   loan_amount_term   600 non-null    float64
 10  credit_history     564 non-null    float64
 11  property_area      614 non-null    object 
 12  loan_status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [55]:
#column name inorder: 
#loan_id,gender,married,dependents,education,self_employed,applicantincome,coapplicantincome,loanamount,loan_amount_term,credit_history,property_area,loan_status

data = pd.read_csv("numerikdataset.csv", skiprows=1, header=None)
data = data.drop(0, axis=1)
data = data.dropna()

m = int(input("615'den küçük M değerini giriniz: "))

data = data.head(m)
data

615'den küçük M değerini giriniz: 300


,1,2,3,4,5,6,7,8,9,10,11,12
1,0.0,1.0,1,1,0.0,4583,1508.0,128.0,360.0,1.0,0.0,0
2,0.0,1.0,0,1,1.0,3000,0.0,66.0,360.0,1.0,1.0,1
3,0.0,1.0,0,0,0.0,2583,2358.0,120.0,360.0,1.0,1.0,1
4,0.0,0.0,0,1,0.0,6000,0.0,141.0,360.0,1.0,1.0,1
5,0.0,1.0,2,1,1.0,5417,4196.0,267.0,360.0,1.0,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
381,0.0,0.0,0,1,0.0,5941,4232.0,296.0,360.0,1.0,0.5,1
382,1.0,0.0,0,1,0.0,6000,0.0,156.0,360.0,1.0,1.0,1
383,0.0,0.0,0,1,1.0,7167,0.0,128.0,360.0,1.0,1.0,1
384,0.0,1.0,2,1,0.0,4566,0.0,100.0,360.0,1.0,1.0,0


## Node class

In [66]:
class Node():
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, info_gain=None, value=None):
        ''' constructor ''' 
        
        # for decision node
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.info_gain = info_gain
        
        # for leaf node
        self.value = value
        

## Tree class

In [57]:
class DecisionTreeClassifier():
    def __init__(self, min_samples_split=2, max_depth=2):
        ''' constructor '''
        
        # initialize the root of the tree 
        self.root = None
        
        # stopping conditions
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        
    def build_tree(self, dataset, curr_depth=0):
        ''' recursive function to build the tree ''' 
        
        X, Y = dataset[:,:-1], dataset[:,-1]
        num_samples, num_features = np.shape(X)
        
        # split until stopping conditions are met
        if num_samples>=self.min_samples_split and curr_depth<=self.max_depth:
            # find the best split
            best_split = self.get_best_split(dataset, num_samples, num_features)
            # check if information gain is positive
            if best_split["info_gain"]>0:
                # recur left
                left_subtree = self.build_tree(best_split["dataset_left"], curr_depth+1)
                # recur right
                right_subtree = self.build_tree(best_split["dataset_right"], curr_depth+1)
                # return decision node
                return Node(best_split["feature_index"], best_split["threshold"], 
                            left_subtree, right_subtree, best_split["info_gain"])
        
        # compute leaf node
        leaf_value = self.calculate_leaf_value(Y)
        # return leaf node
        return Node(value=leaf_value)
    
    def get_best_split(self, dataset, num_samples, num_features):
        ''' function to find the best split '''
        
        # dictionary to store the best split
        best_split = {}
        max_info_gain = -float("inf")
        
        # loop over all the features
        for feature_index in range(num_features):
            feature_values = dataset[:, feature_index]
            possible_thresholds = np.unique(feature_values)
            # loop over all the feature values present in the data
            for threshold in possible_thresholds:
                # get current split
                dataset_left, dataset_right = self.split(dataset, feature_index, threshold)
                # check if childs are not null
                if len(dataset_left)>0 and len(dataset_right)>0:
                    y, left_y, right_y = dataset[:, -1], dataset_left[:, -1], dataset_right[:, -1]
                    # compute information gain
                    curr_info_gain = self.information_gain(y, left_y, right_y, "gini")
                    # update the best split if needed
                    if curr_info_gain>max_info_gain:
                        best_split["feature_index"] = feature_index
                        best_split["threshold"] = threshold
                        best_split["dataset_left"] = dataset_left
                        best_split["dataset_right"] = dataset_right
                        best_split["info_gain"] = curr_info_gain
                        max_info_gain = curr_info_gain
                        
        # return best split
        return best_split
    
    def split(self, dataset, feature_index, threshold):
        ''' function to split the data '''
        
        dataset_left = np.array([row for row in dataset if row[feature_index]<=threshold])
        dataset_right = np.array([row for row in dataset if row[feature_index]>threshold])
        return dataset_left, dataset_right
    
    def information_gain(self, parent, l_child, r_child, mode="entropy"):
        ''' function to compute information gain '''
        
        weight_l = len(l_child) / len(parent)
        weight_r = len(r_child) / len(parent)
        if mode=="gini":
            gain = self.gini_index(parent) - (weight_l*self.gini_index(l_child) + weight_r*self.gini_index(r_child))
        else:
            gain = self.entropy(parent) - (weight_l*self.entropy(l_child) + weight_r*self.entropy(r_child))
        return gain
    
    def entropy(self, y):
        ''' function to compute entropy '''
        
        class_labels = np.unique(y)
        entropy = 0
        for cls in class_labels:
            p_cls = len(y[y == cls]) / len(y)
            entropy += -p_cls * np.log2(p_cls)
        return entropy
    
    def gini_index(self, y):
        ''' function to compute gini index '''
        
        class_labels = np.unique(y)
        gini = 0
        for cls in class_labels:
            p_cls = len(y[y == cls]) / len(y)
            gini += p_cls**2
        return 1 - gini
        
    def calculate_leaf_value(self, Y):
        ''' function to compute leaf node '''
        
        Y = list(Y)
        return max(Y, key=Y.count)
    
    def print_tree(self, tree=None, indent=" "):
        ''' function to print the tree '''
        
        if not tree:
            tree = self.root

        if tree.value is not None:
            print(tree.value)

        else:
            print("X_"+str(tree.feature_index), "<=", tree.threshold, "?", tree.info_gain)
            print("%sleft:" % (indent), end="")
            self.print_tree(tree.left, indent + indent)
            print("%sright:" % (indent), end="")
            self.print_tree(tree.right, indent + indent)
    
    def fit(self, X, Y):
        ''' function to train the tree '''
        
        dataset = np.concatenate((X, Y), axis=1)
        self.root = self.build_tree(dataset)
    
    def predict(self, X):
        ''' function to predict new dataset '''
        
        preditions = [self.make_prediction(x, self.root) for x in X]
        return preditions
    
    def make_prediction(self, x, tree):
        ''' function to predict a single data point '''
        
        if tree.value!=None: return tree.value
        feature_val = x[tree.feature_index]
        if feature_val<=tree.threshold:
            return self.make_prediction(x, tree.left)
        else:
            return self.make_prediction(x, tree.right)

## Train-Test split

In [58]:
X = data.iloc[:, :-1].values
Y = data.iloc[:, -1].values.reshape(-1,1)
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=.2, random_state=41)

## Fit the model

In [72]:
#min_samples_split and max depth can be change.
classifier = DecisionTreeClassifier(min_samples_split=2, max_depth=5)
classifier.fit(X_train,Y_train)
classifier.print_tree()
a = classifier.print_tree()

X_9 <= 0.0 ? 0.0787500000000001
 left:X_7 <= 128.0 ? 0.0634920634920634
  left:0
  right:X_4 <= 0.0 ? 0.10204081632653056
    left:X_5 <= 4300 ? 0.08333333333333331
        left:X_5 <= 2137 ? 0.375
                left:1
                right:0
        right:X_5 <= 4923 ? 0.2222222222222222
                left:1
                right:X_5 <= 14999 ? 0.4444444444444444
                                left:0
                                right:1
    right:0
 right:X_10 <= 0.0 ? 0.01736111111111105
  left:X_6 <= 2840.0 ? 0.044999999999999984
    left:X_1 <= 0.0 ? 0.045867794486215585
        left:X_5 <= 2378 ? 0.11015153984031278
                left:0
                right:X_5 <= 4000 ? 0.07750865051903122
                                left:1
                                right:1
        right:X_5 <= 5042 ? 0.06530612244897954
                left:X_6 <= 1508.0 ? 0.16253968253968257
                                left:0
                                right:1
                right

In [67]:
Y_pred = classifier.predict(X_test) 
from sklearn.metrics import accuracy_score
accuracy_score(Y_test, Y_pred)

0.8

In [ ]:
import graphviz


def visualize_tree(output_string):
    # create a Graph object
    graph = graphviz.Graph(format='png')

    # split the output string into parts separated by '?'
    parts = output_string.strip().split('?')

    # create a queue to hold the nodes and edges
    queue = []

    # loop over the parts
    for part in parts:
        # split the part into the node description and the value
        node_description, value = part.strip().split(' :')

        # create a tuple to hold the node description and value
        node = (node_description, value)

        # add the node to the queue
        queue.append(node)

        # if the queue has more than one element, it means we have a parent and a child
        if len(queue) > 1:
            # get the parent and child nodes
            parent, child = queue[-2], queue[-1]

            # add an edge between the parent and child
            graph.edge(parent[0], child[0])

            # if the child has a value, it means it is a leaf node
            if child[1]:
                # add a label to the leaf node with its value
                graph.node(child[0], label=child[1])
            else:
                # otherwise, it is an internal node and we just add it without a label
                graph.node(child[0])

    # render the graph
    graph.render()
output_string = """
X_9 <= 0.0 ? 0.0787500000000001
 left:X_7 <= 128.0 ? 0.0634920634920634
  left:0
  right:X_4 <= 0.0 ? 0.10204081632653056
    left:X_5 <= 4300 ? 0.08333333333333331
        left:0
        right:1
    right:0
 right:X_10 <= 0.0 ? 0.01736111111111105
  left:X_6 <= 2840.0 ? 0.044999999999999984
    left:X_1 <= 0.0 ? 0.045867794486215585
        left:1
        right:0
    right:X_7 <= 187.0 ? 0.17999999999999994
        left:1
        right:0
  right:X_1 <= 0.0 ? 0.01712155032467544
    left:X_8 <= 360.0 ? 0.05507946598855684
        left:1
        right:0
    right:X_7 <= 255.0 ? 0.020438397581254597
        left:1
        right:1
X_9 <= 0.0 ? 0.0787500000000001
 left:X_7 <= 128.0 ? 0.0634920634920634
  left:0
  right:X_4 <= 0.0 ? 0.10204081632653056
    left:X_5 <= 4300 ? 0.08333333333333331
        left:0
        right:1
    right:0
 right:X_10 <= 0.0 ? 0.01736111111111105
  left:X_6 <= 2840.0 ? 0.044999999999999984
    left:X_1 <= 0.0 ? 0.045867794486215585
        left:1
        right:0
    right:X_7 <= 187.0 ? 0.17999999999999994
        left:1
        right:0
  right:X_1 <= 0.0 ? 0.01712155032467544
    left:X_8 <= 360.0 ? 0.05507946598855684
        left:1
        right:0
    right:X_7 <= 255.0 ? 0.020438397581254597
        left:1
        right:1
"""

visualize_tree(output_string)
